# Рекомендации музыки на Yambda (50M): двухстадийная система

**Задача:** по истории прослушиваний предсказать, какие треки пользователь послушает в ближайшие сутки.

**Структура ноутбука**
0. Настройки и кэш
1. Данные
2. Протокол оценки: временной сплит, метрики
3. EDA (кратко)
4. Бейзлайны: популярное, топ своих треков, затухание
5. iALS (постановка B) и подбор гиперпараметров
6. Обучающие таблицы для ранкера (кандидаты и признаки)
7. Ранкер: validation (YetiRank против классификатора)
8. Ablation
9. Финальный тест (один раз, без подгонки)

**Как работать после перезагрузки :** запустить ячейки 0-2 (кеш).
Тяжёлые блоки закрыты флагами `RUN_*` (по умолчанию `False`).
Порядок первого запуска: блок 1, затем блок 6 (около 25 минут), затем 7-9.

## 0. Настройки и кэш

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"   # до импорта numpy/implicit: иначе iALS сильно тормозит

import json, pickle, time, itertools
import numpy as np
import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from catboost import CatBoostClassifier, CatBoostRanker, Pool

# --- флаги тяжёлых блоков ---
RUN_EDA = False
RUN_BASELINES = False       # быстро, но печатает много
RUN_ALS_GRID = False        # ~40+ минут
RUN_BUILD_TABLES = False    # ~5 минут на отсечку (iALS), всего ~25 минут
RUN_VALIDATION_RANKER = False
RUN_ABLATION = False        # ~15-20 минут
RUN_FINAL_TEST = False      # ~15-20 минут, запускать один раз

# --- константы ---
DAY = 86400                 # время в секундах
GAP = 30 * 60               # зазор между train и целевыми сутками
TAU = 14                    # память затухания (дни), выбрана на validation
K_HIST, K_ALS, K_POP = 500, 100, 100
ALS_PARAMS = dict(factors=256, alpha=0.3, reg=0.01, iters=15)
NEG_KEEP = 0.25             # доля негативов при обучении ранкера
os.makedirs("results", exist_ok=True)

## 1. Данные

Yambda 50M, таблица `listens`. Положительный пример: прослушано не менее 50% трека.
Путь к файлу на Hugging Face сверь через `list_repo_files("yandex/yambda", repo_type="dataset")`.

In [ ]:
if os.path.exists("pos.parquet") and os.path.exists("meta.json"):
    pos = pd.read_parquet("pos.parquet")
    t_max = json.load(open("meta.json"))["t_max"]
else:
    from huggingface_hub import hf_hub_download
    path = hf_hub_download("yandex/yambda", "flat/50m/listens.parquet", repo_type="dataset")
    listens = pd.read_parquet(path)
    t_max = int(listens["timestamp"].max())
    pos = listens.loc[listens["played_ratio_pct"] >= 50, ["uid", "item_id", "timestamp"]].astype("int64")
    pos.to_parquet("pos.parquet")
    json.dump({"t_max": t_max}, open("meta.json", "w"))
    del listens

pos = pos.astype("int64")   # беззнаковый тип ломал exp(): приводим один раз
test_start = t_max - DAY
print("pos:", pos.shape, "| t_max:", t_max, "| test_start:", test_start)

## 2. Протокол оценки

- **Отсечка (cutoff):** признаки считаются по данным строго до `cutoff - GAP`, цель: сутки `[cutoff, cutoff + DAY)`.
- **Тест:** `cutoff = test_start`. **Validation:** `cutoff = test_start - s * DAY`, `s = 1, 2, 3` (усреднение).
- Метрики: Recall@100 и NDCG@100, усреднение по пользователям, `truth` полный (постановка A).
- Постановка B («новое»): из `truth` убраны треки, слышанные в train.

In [ ]:
def split_at(pos, cutoff):
    train = pos[pos["timestamp"] < cutoff - GAP]
    target = pos[(pos["timestamp"] >= cutoff) & (pos["timestamp"] < cutoff + DAY)]
    truth = target.groupby("uid")["item_id"].apply(set).to_dict()
    users = set(train["uid"])
    truth = {u: s for u, s in truth.items() if u in users}   # холодный старт почти нулевой
    return train, truth

def recall_at_k(recs, truth, k=100):
    return np.mean([len(set(recs.get(u, [])[:k]) & t) / len(t) for u, t in truth.items()])

def ndcg_at_k(recs, truth, k=100):
    out = []
    for u, t in truth.items():
        r = recs.get(u, [])[:k]
        dcg = sum(1 / np.log2(i + 2) for i, it in enumerate(r) if it in t)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(t), k)))
        out.append(dcg / idcg)
    return np.mean(out)

def rank_metrics(df, score, truth, k=100):
    """df: колонки uid, item_id; score: оценка для каждой строки."""
    d = (df[["uid", "item_id"]].assign(s=score)
           .sort_values(["uid", "s"], ascending=[True, False])
           .groupby("uid").head(k))
    recs = d.groupby("uid")["item_id"].apply(list).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

def cand_recall(cands, truth):
    return np.mean([len(set(cands.get(u, [])) & t) / len(t) for u, t in truth.items()])

## 3. EDA (кратко)

In [ ]:
if RUN_EDA:
    print("пользователей:", pos["uid"].nunique(), "| треков:", pos["item_id"].nunique(), "| событий:", len(pos))
    print("период (суток):", (pos["timestamp"].max() - pos["timestamp"].min()) / DAY)
    print(pos.groupby("uid").size().describe())
    print(pos.groupby("item_id").size().describe())
    train, truth = split_at(pos, test_start)
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    n_all = sum(len(t) for t in truth.values())
    n_new = sum(len(t - seen[u]) for u, t in truth.items())
    print("доля повторов среди (user, track) в тесте:", 1 - n_new / n_all)

## 4. Бейзлайны

- `eval_popular`: популярное за последние `days` дней (**жёсткое окно**, не затухание).
- `eval_my_top`: свои треки по числу прослушиваний.
- `eval_my_top_decay`: свои треки с весом `sum(exp(-dt/tau))` (настоящее затухание).

In [ ]:
def eval_popular(pos, cutoff, days=None, k=100):
    train, truth = split_at(pos, cutoff)
    if days is not None:
        train = train[train["timestamp"] >= cutoff - GAP - days * DAY]
    top = train["item_id"].value_counts().index[:k].tolist()
    recs = {u: top for u in truth}
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

def eval_popular_new(pos, cutoff, days=1, k=100):
    train, truth = split_at(pos, cutoff)
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    truth_new = {u: t - seen[u] for u, t in truth.items()}
    truth_new = {u: t for u, t in truth_new.items() if t}
    recent = train[train["timestamp"] >= cutoff - GAP - days * DAY]
    top = recent["item_id"].value_counts().index[:1000].tolist()
    recs = {u: [i for i in top if i not in seen[u]][:k] for u in truth_new}
    return recall_at_k(recs, truth_new, k), ndcg_at_k(recs, truth_new, k)

def eval_my_top(pos, cutoff, k=100):
    train, truth = split_at(pos, cutoff)
    top = (train.groupby(["uid", "item_id"]).size().reset_index(name="n")
                .sort_values(["uid", "n"], ascending=[True, False]))
    recs = top.groupby("uid")["item_id"].apply(lambda s: s.tolist()[:k]).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

def eval_my_top_decay(pos, cutoff, tau_days, k=100):
    train, truth = split_at(pos, cutoff)
    dt = float(cutoff - GAP) - train["timestamp"].values.astype("float64")
    w = np.exp(-dt / (tau_days * DAY))
    assert w.max() <= 1.0 and np.isfinite(w).all()
    s = (train[["uid", "item_id"]].assign(w=w)
              .groupby(["uid", "item_id"], as_index=False)["w"].sum()
              .sort_values(["uid", "w"], ascending=[True, False]))
    recs = s.groupby("uid")["item_id"].apply(lambda x: x.tolist()[:k]).to_dict()
    return recall_at_k(recs, truth, k), ndcg_at_k(recs, truth, k)

SHIFTS_VAL = [1, 2, 3]
def avg_over_shifts(fn, *args, **kw):
    return np.mean([fn(pos, test_start - s * DAY, *args, **kw) for s in SHIFTS_VAL], axis=0)

In [ ]:
if RUN_BASELINES:
    for days in [None, 0.25, 0.5, 1, 3, 7, 14, 30]:
        print("popular window", days, avg_over_shifts(eval_popular, days))
    print("my_top (счётчик):", avg_over_shifts(eval_my_top))
    for tau in [0.5, 1, 3, 7, 14, 30, 100]:
        print("my_top_decay tau", tau, avg_over_shifts(eval_my_top_decay, tau))
    print("popular без повторов (B):", avg_over_shifts(eval_popular_new, 1))

## 5. iALS (постановка B)

Только новые для пользователя треки (`filter_already_liked_items=True`), `truth_new` без слышанного.
В итоговой схеме iALS даёт кандидатов-новинки и признаки `als_rank`, `als_score`.

In [ ]:
def build_matrix(train):
    counts = train.groupby(["uid", "item_id"]).size().reset_index(name="n")
    u_cat = counts["uid"].astype("category")
    i_cat = counts["item_id"].astype("category")
    mat = sp.csr_matrix((counts["n"].astype("float32"), (u_cat.cat.codes, i_cat.cat.codes)),
                        shape=(len(u_cat.cat.categories), len(i_cat.cat.categories)))
    return mat, list(u_cat.cat.categories), list(i_cat.cat.categories)

def fit_als(train, factors, alpha, reg, iters, seed=42):
    mat, user_ids, item_ids = build_matrix(train)
    mat.data = 1 + alpha * mat.data          # уверенность: 1 + alpha * число прослушиваний
    model = AlternatingLeastSquares(factors=factors, regularization=reg,
                                    iterations=iters, random_state=seed)
    model.fit(mat)
    return model, mat, user_ids, item_ids

def eval_als(pos, cutoff, factors=256, reg=0.01, iters=15, alpha=0.3, k=100):
    train, truth = split_at(pos, cutoff)
    seen = train.groupby("uid")["item_id"].apply(set).to_dict()
    truth_new = {u: t - seen[u] for u, t in truth.items()}
    truth_new = {u: t for u, t in truth_new.items() if t}
    model, mat, user_ids, item_ids = fit_als(train, factors, alpha, reg, iters)
    u_index = {u: i for i, u in enumerate(user_ids)}
    users = [u for u in truth_new if u in u_index]
    idx = np.array([u_index[u] for u in users])
    rec_idx, _ = model.recommend(idx, mat[idx], N=k, filter_already_liked_items=True)
    recs = {u: [item_ids[j] for j in row] for u, row in zip(users, rec_idx)}
    truth_new = {u: truth_new[u] for u in users}
    return recall_at_k(recs, truth_new, k), ndcg_at_k(recs, truth_new, k)

if RUN_ALS_GRID:
    rows = []
    for factors, alpha, reg in itertools.product([32, 64, 128], [1, 10, 40], [0.01, 0.1]):
        r, n = np.mean([eval_als(pos, test_start - s * DAY, factors=factors, reg=reg, alpha=alpha)
                        for s in SHIFTS_VAL], axis=0)
        rows.append(dict(factors=factors, alpha=alpha, reg=reg, recall=r, ndcg=n))
        pd.DataFrame(rows).to_csv("results/als_grid_partial.csv", index=False)
        print(rows[-1])
    pd.DataFrame(rows).sort_values("ndcg", ascending=False).to_csv("results/als_grid.csv", index=False)
    # затем добор: (256, 1), (256, 0.3), (128, 0.3), (128, 0.1), (512, 0.3), reg=0.01

## 6. Обучающие таблицы для ранкера

Одна строка = пара (пользователь, кандидат) на конкретной отсечке. Признаки только по данным до отсечки,
метка `y` = трек есть в `truth` следующих суток.

Кандидаты: топ-500 истории по `w14`, топ-100 iALS-новинок, топ-100 популярного за сутки.
Таблицы для `shift = 0..4` сохраняются в parquet, `truth` в pickle (shift 0 это тест).

In [ ]:
def build_table(pos, cutoff, k_hist=K_HIST, k_als=K_ALS, k_pop=K_POP,
                factors=ALS_PARAMS["factors"], alpha=ALS_PARAMS["alpha"]):
    train, truth = split_at(pos, cutoff)
    train = train[["uid", "item_id", "timestamp"]].astype("int64")
    users = np.array(list(truth), dtype="int64")
    t_ref = float(cutoff - GAP)
    age_all = t_ref - train["timestamp"].values.astype("float64")

    # признаки трека
    item_feat = pd.DataFrame({
        "pop_all": train.groupby("item_id").size(),
        "pop_7d":  train[age_all <= 7 * DAY].groupby("item_id").size(),
        "pop_1d":  train[age_all <= DAY].groupby("item_id").size(),
    }).fillna(0)

    # признаки пары (пользователь, трек) из истории
    tu = train[train["uid"].isin(users)]
    age = t_ref - tu["timestamp"].values.astype("float64")
    ui = pd.DataFrame({"uid": tu["uid"].values, "item_id": tu["item_id"].values, "age": age})
    for t in [1, 7, 14, 30, 100]:
        ui[f"w{t}"] = np.exp(-age / (t * DAY))
    ui["n7"] = (age <= 7 * DAY).astype("int32")
    g = ui.groupby(["uid", "item_id"]).agg(
        n=("age", "size"), last_days=("age", "min"), first_days=("age", "max"),
        n7=("n7", "sum"), w1=("w1", "sum"), w7=("w7", "sum"), w14=("w14", "sum"),
        w30=("w30", "sum"), w100=("w100", "sum")).reset_index()
    g["last_days"] /= DAY
    g["first_days"] /= DAY
    g["hist_rank"] = g.groupby("uid")["w14"].rank(ascending=False, method="first")

    # признаки пользователя
    user_feat = pd.DataFrame({
        "u_n": ui.groupby("uid").size(),
        "u_n7": ui.groupby("uid")["n7"].sum(),
        "u_distinct": g.groupby("uid").size(),
    })

    # кандидаты: iALS (только новое для пользователя)
    model, mat, user_ids, item_ids = fit_als(train, factors, alpha, 0.01, 15)
    u_index = {u: i for i, u in enumerate(user_ids)}
    idx = np.array([u_index[u] for u in users])
    rec_idx, rec_sc = model.recommend(idx, mat[idx], N=k_als, filter_already_liked_items=True)
    item_arr = np.array(item_ids, dtype="int64")
    als = pd.DataFrame({"uid": np.repeat(users, k_als),
                        "item_id": item_arr[rec_idx].ravel(),
                        "als_rank": np.tile(np.arange(1, k_als + 1), len(users)),
                        "als_score": rec_sc.ravel()})

    # кандидаты: свежее популярное
    top = train.loc[age_all <= DAY, "item_id"].value_counts().index[:k_pop].to_numpy()
    pop = pd.DataFrame({"uid": np.repeat(users, len(top)), "item_id": np.tile(top, len(users))})

    # объединение и признаки
    cand = pd.concat([g.loc[g["hist_rank"] <= k_hist, ["uid", "item_id"]],
                      als[["uid", "item_id"]], pop]).drop_duplicates()
    cand = (cand.merge(g, on=["uid", "item_id"], how="left")
                .merge(als, on=["uid", "item_id"], how="left")
                .merge(item_feat, left_on="item_id", right_index=True, how="left")
                .merge(user_feat, left_on="uid", right_index=True, how="left"))
    for c in ["n", "n7", "w1", "w7", "w14", "w30", "w100"]:
        cand[c] = cand[c].fillna(0)
    cand["n_share"] = cand["n"] / cand["u_n"]

    # метка
    lab = pd.DataFrame([(u, i) for u, s in truth.items() for i in s],
                       columns=["uid", "item_id"]).astype("int64")
    lab["y"] = 1
    cand = cand.merge(lab, how="left", on=["uid", "item_id"])
    cand["y"] = cand["y"].fillna(0).astype("int8")

    fl = cand.select_dtypes("float64").columns
    cand[fl] = cand[fl].astype("float32")
    return cand, truth

def candidate_recall_from_table(cand, truth):
    hits = cand.groupby("uid")["y"].sum()
    n_truth = pd.Series({u: len(t) for u, t in truth.items()})
    n_truth.index = n_truth.index.astype("int64")
    return (hits.reindex(n_truth.index).fillna(0) / n_truth).mean()

if RUN_BUILD_TABLES:
    for s in [1, 2, 3, 4, 0]:
        if os.path.exists(f"table_shift{s}.parquet"):
            continue                         # уже посчитано
        cand, truth = build_table(pos, test_start - s * DAY)
        cand.to_parquet(f"table_shift{s}.parquet")
        pickle.dump(truth, open(f"truth_shift{s}.pkl", "wb"))
        print("shift", s, cand.shape, "кандидатов на пользователя:",
              round(len(cand) / cand["uid"].nunique()),
              "| потолок (candidate recall):", round(candidate_recall_from_table(cand, truth), 4))

## 7. Ранкер: validation

Обучение на shift 2-4, проверка на shift 1. Сравниваем сортировку по `w14`, классификатор и YetiRank.
В обучении: все `y=1` и 25% негативов. Проверка идёт по всем кандидатам.

In [ ]:
def load_tables(shifts):
    return pd.concat([pd.read_parquet(f"table_shift{s}.parquet").assign(shift=s) for s in shifts],
                     ignore_index=True)

def feature_list(df):
    return [c for c in df.columns if c not in ("uid", "item_id", "y", "shift", "group")]

def subsample_negatives(df, keep=NEG_KEEP, seed=42):
    rng = np.random.RandomState(seed)
    return df[(df["y"].values == 1) | (rng.rand(len(df)) < keep)].copy()

def make_ranker_pools(tr, va, features):
    tr = tr.copy()
    tr["group"] = tr["shift"].astype(str) + "_" + tr["uid"].astype(str)   # группа = (отсечка, пользователь)
    tr = tr.sort_values("group", kind="stable").reset_index(drop=True)
    va = va.sort_values("uid", kind="stable").reset_index(drop=True)
    return (tr, va,
            Pool(tr[features], tr["y"], group_id=tr["group"].values),
            Pool(va[features], va["y"], group_id=va["uid"].values))

if RUN_VALIDATION_RANKER:
    tr = load_tables([2, 3, 4]); va = pd.read_parquet("table_shift1.parquet")
    truth_va = pickle.load(open("truth_shift1.pkl", "rb"))
    FEATURES = feature_list(tr); assert len(FEATURES) == 19
    print("sort by w14:", rank_metrics(va, va["w14"].values, truth_va))

    clf = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, random_seed=42, verbose=100)
    clf.fit(tr[FEATURES], tr["y"])
    print("CatBoostClassifier:", rank_metrics(va, clf.predict_proba(va[FEATURES])[:, 1], truth_va))

    tr_s = subsample_negatives(tr)
    tr_s, va_s, pool_tr, pool_va = make_ranker_pools(tr_s, va, FEATURES)
    rk = CatBoostRanker(loss_function="YetiRank", iterations=200, learning_rate=0.15,
                        depth=6, random_seed=42, verbose=50)
    rk.fit(pool_tr)
    print("YetiRank:", rank_metrics(va_s, rk.predict(pool_va), truth_va))

## 8. Ablation

Классификатор, 200 деревьев, lr=0.15, одинаковые настройки во всех запусках, validation shift 1.
Шум оцениваем по двум seed'ам полной модели: разницы порядка 0.001 нельзя считать эффектом.

In [ ]:
ABLATION_GROUPS = {
    "no_ials":       ["als_rank", "als_score"],
    "no_popularity": ["pop_all", "pop_7d", "pop_1d"],
    "no_multi_tau":  ["w1", "w7", "w30", "w100"],
    "no_user_feats": ["u_n", "u_n7", "u_distinct"],
    "no_hist_extra": ["hist_rank", "n_share", "first_days", "n", "n7"],
}

if RUN_ABLATION:
    tr = load_tables([2, 3, 4]); va = pd.read_parquet("table_shift1.parquet")
    truth_va = pickle.load(open("truth_shift1.pkl", "rb"))
    FEATURES = feature_list(tr)

    def run_ablation(drop, seed=42):
        feats = [f for f in FEATURES if f not in drop]
        m = CatBoostClassifier(iterations=200, learning_rate=0.15, depth=6, random_seed=seed, verbose=0)
        m.fit(tr[feats], tr["y"])
        return rank_metrics(va, m.predict_proba(va[feats])[:, 1], truth_va)

    experiments = [("full_seed42", [], 42), ("full_seed43", [], 43)] + \
                  [(n, d, 42) for n, d in ABLATION_GROUPS.items()]
    rows = []
    for name, drop, seed in experiments:
        r, n = run_ablation(drop, seed)
        rows.append(dict(experiment=name, dropped=",".join(drop), recall=r, ndcg=n))
        print(rows[-1])
        pd.DataFrame(rows).to_csv("results/ablation_partial.csv", index=False)
    abl = pd.DataFrame(rows)
    abl["d_ndcg_vs_full"] = abl["ndcg"] - abl.loc[0, "ndcg"]
    abl.to_csv("results/ablation.csv", index=False)
    print(abl)

## 9. Финальный тест

Конфигурация зафиксирована **до** взгляда на тест: YetiRank, 200 деревьев, lr=0.15, depth=6, 25% негативов.
Обучение на shift 1-4, три seed'а, результат: среднее и разброс. Лучший seed не выбираем.
Запускать один раз.

In [ ]:
if RUN_FINAL_TEST:
    t0 = time.time()
    tr = load_tables([1, 2, 3, 4])
    te = pd.read_parquet("table_shift0.parquet")
    truth_te = pickle.load(open("truth_shift0.pkl", "rb"))
    FEATURES = feature_list(tr); assert len(FEATURES) == 19

    print("candidate recall (потолок), тест:", round(candidate_recall_from_table(te, truth_te), 4))

    tr_s = subsample_negatives(tr)
    tr_s, te_s, pool_tr, pool_te = make_ranker_pools(tr_s, te, FEATURES)

    results = {"sort_by_w14": rank_metrics(te_s, te_s["w14"].values, truth_te)}
    for seed in [42, 43, 44]:
        rk = CatBoostRanker(loss_function="YetiRank", iterations=200, learning_rate=0.15,
                            depth=6, random_seed=seed, verbose=0)
        rk.fit(pool_tr)
        results[f"yetirank_seed{seed}"] = rank_metrics(te_s, rk.predict(pool_te), truth_te)
        print(seed, results[f"yetirank_seed{seed}"], f"{time.time() - t0:.0f}s")

    yr = np.array([v for k, v in results.items() if k.startswith("yetirank")])
    print("YetiRank среднее:", yr.mean(axis=0), "std:", yr.std(axis=0))

    # одиночные бейзлайны на тесте
    results["popular_1d"] = eval_popular(pos, test_start, 1)
    results["my_top_counter"] = eval_my_top(pos, test_start)
    results["my_top_decay_tau14"] = eval_my_top_decay(pos, test_start, TAU)
    pd.DataFrame(results, index=["recall", "ndcg"]).T.to_csv("results/test_results.csv")

    imp = pd.Series(rk.get_feature_importance(type="PredictionValuesChange"), index=FEATURES)
    imp.sort_values(ascending=False).to_csv("results/importance_ranker.csv")
    print(pd.DataFrame(results, index=["recall", "ndcg"]).T)

## Итоги и ограничения

Результаты для README лежат в `results/`: `ablation.csv`, `test_results.csv`, `importance_ranker.csv`, `als_grid.csv`.

**Ограничения:** офлайн-метрики измеряют угадывание поведения, а не пользу рекомендаций (нужен A/B-тест);
один тестовый день и около 4.6k пользователей; ablation сделан на классификаторе и одной отсечке;
лайки, дизлайки, аудио-эмбеддинги и артисты не использовались.